# CropCare AI: Step 3 (stronger model) on Kaggle

Builds **Experiment C** and compares backbones on it:

- more field photos: **PlantDoc + PlantWild** (in-the-wild web photos) + **rice** + **wheat** field datasets
- **near-duplicate removal**: training photos that are re-saved copies of any test photo are dropped
- **source-balanced sampling** so lab photos do not drown out field photos
- **288 px** images and **test-time flip averaging**
- backbones: ConvNeXt-Tiny (main), DINOv2-Small, EfficientNetV2-B0, MobileNetV3 (phone-sized)

**Before running:**
1. *Settings → Accelerator → GPU T4 x2*, *Internet → On*
2. *Add Input* (search the name, click +):
   - `abdallahalidev/plantvillage-dataset` (PlantVillage)
   - `nirmalsankalana/rice-leaf-disease-image` (rice)
   - `freedomfighter1290/wheat-disease` (wheat)
3. Run with **Save Version → Save & Run All (Commit)** so it keeps running with the tab closed.

Everything large is kept in `/kaggle/temp`; only the results zip goes to Output.

In [ ]:
import glob, os, subprocess, shutil

def sh(cmd, required=True):
    """Run a shell command, show its output, stop the notebook only if it is required."""
    print(">>", cmd)
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        if required:
            raise RuntimeError(f"Command failed: {cmd}")
        print(f"(optional step failed, continuing)")
    return result.returncode == 0

os.makedirs("/kaggle/temp", exist_ok=True)
sh("rm -rf /kaggle/temp/repo && git clone -q https://github.com/MohammadAftabAlam/Crop-Disease-Detection.git /kaggle/temp/repo")
os.chdir("/kaggle/temp/repo/ai-service")
sh("pip install -q timm segmentation-models-pytorch pydantic-settings onnx onnxruntime")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU accelerator")

## 1. Datasets

In [ ]:
def find_dataset_root(marker_folder):
    hits = glob.glob(f"/kaggle/input/**/{marker_folder}", recursive=True)
    hits = [h for h in hits if "segmented" not in h.lower() and "grayscale" not in h.lower()]
    return os.path.dirname(sorted(hits, key=len)[0]) if hits else None

def find_input(*keywords):
    """Top folder of an attached Kaggle dataset whose name contains any keyword."""
    for path in sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*/*")):
        if any(k in os.path.basename(path).lower() for k in keywords):
            return path
    return None

PLANTVILLAGE = find_dataset_root("Tomato___Late_blight") or find_dataset_root("Tomato_Late_blight")
RICE = find_input("rice")
WHEAT = find_input("wheat")
print("PlantVillage:", PLANTVILLAGE)
print("Rice:", RICE)
print("Wheat:", WHEAT)

sh("git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git /kaggle/temp/PlantDoc-Dataset")
# PlantWild v1 (CC BY-NC-ND 4.0, academic use), 2.7 GB
sh("wget -q -O /kaggle/temp/plantwild.zip https://huggingface.co/datasets/uqtwei2/PlantWild/resolve/main/plantwild.zip"
   " && unzip -q -o /kaggle/temp/plantwild.zip -d /kaggle/temp/plantwild && rm /kaggle/temp/plantwild.zip", required=False)

## 2. Manifests
Each command prints how many images were found per split and which folders were skipped
(classes we do not cover, e.g. apple or tungro). That is expected.

In [ ]:
P = "python -m cropcare_ai.data.prepare"
sh(f'{P} --name plantvillage --root "{PLANTVILLAGE}"')
sh(f'{P} --name plantdoc --root /kaggle/temp/PlantDoc-Dataset --layout keep --val-from-train 0.2')
sh(f'{P} --name plantwild --root /kaggle/temp/plantwild --search', required=False)
if RICE:
    sh(f'{P} --name rice --root "{RICE}" --crop rice --search --val-from-train 0.2', required=False)
if WHEAT:
    sh(f'{P} --name wheat --root "{WHEAT}" --crop wheat --search --val-from-train 0.2', required=False)
sh("ls -la data/manifests")

## 3. Train, calibrate and evaluate
`RUN` lists the backbones to train. ConvNeXt-Tiny is the main model; remove entries if GPU time is short
(rough T4 times: ConvNeXt ~60 min, DINOv2 ~60 min, EfficientNetV2 ~35 min, MobileNetV3 ~25 min).

In [ ]:
RUN = ["c_convnext_tiny", "c_mobilenetv3", "c_efficientnetv2_b0", "c_dinov2_small"]

CALIBRATION = [m for m in ["data/manifests/plantdoc.csv", "data/manifests/rice.csv", "data/manifests/wheat.csv"]
               if os.path.exists(m)]
TEST_SETS = [  # (manifest, name, description)
    ("data/manifests/plantvillage.csv", "plantvillage_test", "PlantVillage test (lab photos)"),
    ("data/manifests/plantdoc.csv", "plantdoc_test", "PlantDoc test (field photos)"),
    ("data/manifests/plantwild.csv", "plantwild_test", "PlantWild test (web field photos)"),
    ("data/manifests/rice.csv", "rice_test", "Rice field photos test"),
    ("data/manifests/wheat.csv", "wheat_test", "Wheat field photos test"),
]

def run_experiment(name):
    model_dir = f"artifacts/{name}"
    sh(f"python -m cropcare_ai.training.train_classifier --config configs/train/{name}.yaml")
    manifests = " ".join(f"--manifest {m}" for m in CALIBRATION)
    sh(f"python -m cropcare_ai.training.calibrate --model-dir {model_dir} {manifests} --split val --tta")
    for manifest, test_name, description in TEST_SETS:
        if os.path.exists(manifest):
            sh(f'python -m cropcare_ai.training.evaluate --model-dir {model_dir} --manifest {manifest} '
               f'--name {test_name} --description "{description}"', required=False)

# Zip each finished model into Output right away, so a 12-hour timeout cannot lose it
def save_now(name):
    sh(f"cd artifacts && zip -qr /kaggle/working/{name}.zip {name} -x '{name}/*.onnx'", required=False)

for name in RUN:
    try:
        run_experiment(name)
        save_now(name)
    except RuntimeError as error:
        print(f"!! {name} failed: {error}. Continuing with the next model.")

## 4. Comparison table

In [ ]:
done = [f"artifacts/{n}" for n in RUN if os.path.exists(f"artifacts/{n}/bundle.json")]
if not done:
    raise RuntimeError("No model finished training; scroll up to the first error.")
sh("python -m cropcare_ai.training.report --model-dir " + " ".join(done))

## 5. Export for phones/offline and download

In [ ]:
for name in ["c_mobilenetv3", "c_convnext_tiny"]:
    if os.path.exists(f"artifacts/{name}/bundle.json"):
        sh(f"python -m cropcare_ai.training.export_onnx --model-dir artifacts/{name}", required=False)

sh("cp data/manifests/*.summary.json artifacts/ 2>/dev/null", required=False)
sh(f"cp {done[0]}/report.md /kaggle/working/report.md", required=False)
# Zip only the Step 3 folders; mobilenet's int8 ONNX is kept (small, for offline use)
# Per-model zips were saved as each model finished; add the ONNX exports and dataset summaries now
for name in ["c_mobilenetv3"]:
    if os.path.exists(f"artifacts/{name}/model.int8.onnx"):
        sh(f"cd artifacts && zip -qr /kaggle/working/{name}.zip {name}/model.int8.onnx {name}/onnx_export.json", required=False)
sh("cd artifacts && zip -qr /kaggle/working/dataset_summaries.zip *.summary.json", required=False)
sh("ls -lh /kaggle/working")
print("Download the c_*.zip files and report.md from the Output tab; unzip each into ai-service/artifacts/.")